# Packet P-032 — Subgroup Fairness Across Composition Families

**Decision question:** Does the model rank equally well within each composition family, or does it systematically fail on some families while appearing adequate in aggregate?

**Fastest falsifier:** Within-family tau-b for the smallest families (Pure FA, FA-Cs). If tau-b < 0.05, subgroup fairness is falsified.

**Success:** All families with n ≥ 40 have mean within-family tau-b > 0.10.
**Kill:** Any family with n ≥ 40 has mean within-family tau-b indistinguishable from zero (95% CI crosses zero).

In [1]:
"""Cell 1 — Load data, classify families, setup."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

def classify_family(row):
    ma = row["MA"] > 0
    fa = row["FA"] > 0
    cs = row["Cs"] > 0
    if ma and not fa and not cs:
        return "Pure MA"
    elif fa and not ma and not cs:
        return "Pure FA"
    elif ma and fa and not cs:
        return "MA-FA mixed"
    elif fa and cs and not ma:
        return "FA-Cs"
    elif ma and fa and cs:
        return "Triple cation"
    else:
        return "Other"

df["family"] = df.apply(classify_family, axis=1)
family_counts = df["family"].value_counts()
print("── Composition family counts ──")
print(family_counts.to_string())
print(f"\nTotal devices: {len(df)}")

── Composition family counts ──
family
Pure MA          967
Triple cation    197
MA-FA mixed      197
Other             88
Pure FA           50
FA-Cs             44

Total devices: 1543


In [2]:
"""Cell 2 — Within-family tau-b across 20 splits."""

N_SPLITS = 20
families_to_test = [f for f in family_counts.index if family_counts[f] >= 30]

# Store per-family tau-b across splits
family_taus = {fam: [] for fam in families_to_test}
overall_taus = []

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    # Overall tau-b
    tau_all, _ = kendalltau(y_te, preds)
    overall_taus.append(tau_all)
    
    # Within-family tau-b
    test_families = df.loc[X_te.index, "family"]
    for fam in families_to_test:
        fam_mask = test_families == fam
        if fam_mask.sum() >= 5:  # need enough devices
            y_fam = y_te[fam_mask]
            p_fam = preds[fam_mask.values]
            tau_fam, _ = kendalltau(y_fam, p_fam)
            family_taus[fam].append(tau_fam)

    if (seed + 1) % 10 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Report
print(f"\n{'=' * 70}")
print(f"WITHIN-FAMILY RANKING QUALITY")
print(f"{'=' * 70}")
print(f"{'Family':<20} {'n':>5} {'Mean tau-b':>11} {'Std':>8} {'95% CI':>20} {'Splits':>7}")
print("-" * 70)

summary_rows = []
for fam in families_to_test:
    taus = family_taus[fam]
    n = family_counts[fam]
    if len(taus) >= 3:
        mean_t = np.mean(taus)
        std_t = np.std(taus)
        ci_lo = np.percentile(taus, 2.5)
        ci_hi = np.percentile(taus, 97.5)
        print(f"{fam:<20} {n:>5} {mean_t:>11.4f} {std_t:>8.4f} [{ci_lo:>7.4f}, {ci_hi:>6.4f}] {len(taus):>7}")
        summary_rows.append({
            'family': fam, 'n_devices': n, 'n_splits': len(taus),
            'mean_tau': mean_t, 'std_tau': std_t, 'ci_lo': ci_lo, 'ci_hi': ci_hi
        })
    else:
        print(f"{fam:<20} {n:>5}   (too few test appearances)")

print("-" * 70)
print(f"{'Overall':<20} {len(df):>5} {np.mean(overall_taus):>11.4f} {np.std(overall_taus):>8.4f}")

summary_df = pd.DataFrame(summary_rows)

  Completed 10/20 splits


  Completed 20/20 splits

WITHIN-FAMILY RANKING QUALITY
Family                   n  Mean tau-b      Std               95% CI  Splits
----------------------------------------------------------------------
Pure MA                967      0.2780   0.0480 [ 0.1831, 0.3495]      20
Triple cation          197      0.4012   0.0921 [ 0.2282, 0.5563]      20
MA-FA mixed            197      0.3289   0.0920 [ 0.1646, 0.4781]      20
Other                   88      0.1893   0.1390 [-0.0762, 0.4156]      20
Pure FA                 50      0.0240   0.2020 [-0.3305, 0.3667]      20
FA-Cs                   44      0.3230   0.2370 [-0.0480, 0.7594]      19
----------------------------------------------------------------------
Overall               1543      0.2999   0.0420


In [3]:
"""Cell 3 — Evaluate thresholds and save."""

# Success: all families with n >= 40 have mean within-family tau-b > 0.10
# Kill: any family with n >= 40 has CI crossing zero

big_families = summary_df[summary_df['n_devices'] >= 40]
ci_crosses_zero = big_families[big_families['ci_lo'] <= 0]
below_threshold = big_families[big_families['mean_tau'] < 0.10]

print("── Evaluation ──\n")
print(f"Families with n >= 40: {len(big_families)}")
print(f"Families with CI crossing zero: {len(ci_crosses_zero)}")
if len(ci_crosses_zero) > 0:
    for _, r in ci_crosses_zero.iterrows():
        print(f"  ⚠ {r['family']}: tau-b {r['mean_tau']:.4f}, CI [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]")

print(f"Families with mean tau-b < 0.10: {len(below_threshold)}")
if len(below_threshold) > 0:
    for _, r in below_threshold.iterrows():
        print(f"  ⚠ {r['family']}: tau-b {r['mean_tau']:.4f}")

# Disparity ratio: worst family / overall
if len(big_families) > 0:
    worst_tau = big_families['mean_tau'].min()
    overall_mean = np.mean(overall_taus)
    disparity = worst_tau / overall_mean if overall_mean > 0 else 0
    worst_fam = big_families.loc[big_families['mean_tau'].idxmin(), 'family']
    print(f"\nWorst family: {worst_fam} (tau-b {worst_tau:.4f})")
    print(f"Disparity ratio (worst/overall): {disparity:.2f}")

# Status determination
if len(ci_crosses_zero) > 0:
    status = "Negative"
elif len(below_threshold) > 0:
    status = "Promising"
else:
    status = "Confirmed"

# Save
summary_df.to_csv('Packet_P032_Subgroup_Fairness.csv', index=False)
print(f"\nSaved: Packet_P032_Subgroup_Fairness.csv")

print(f"\n{'=' * 60}")
print(f"P-032 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Model ranks well within all major composition families.")
elif status == "Promising":
    print("Some families have weak but non-zero ranking quality.")
else:
    print("Model fails to rank within at least one major family.")
    print("Lab partners should be warned about family-specific limitations.")

── Evaluation ──

Families with n >= 40: 6
Families with CI crossing zero: 3
  ⚠ Other: tau-b 0.1893, CI [-0.0762, 0.4156]
  ⚠ Pure FA: tau-b 0.0240, CI [-0.3305, 0.3667]
  ⚠ FA-Cs: tau-b 0.3230, CI [-0.0480, 0.7594]
Families with mean tau-b < 0.10: 1
  ⚠ Pure FA: tau-b 0.0240

Worst family: Pure FA (tau-b 0.0240)
Disparity ratio (worst/overall): 0.08

Saved: Packet_P032_Subgroup_Fairness.csv

P-032 STATUS: Negative
Model fails to rank within at least one major family.
Lab partners should be warned about family-specific limitations.
